# 01 — Data Quality Audit

Audit the supplied aggregate HHS source without mutating it, then compare the
raw findings with the production preprocessing report. Logical anomalies remain
visible by design; an audit failure is not the same as a pipeline failure.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
from app_utils import DATE_COLUMN
from src.preprocessor import preprocess_data
from src.validation import validate_capacity_data

raw_path = RAW_DIR / "HHS_Unaccompanied_Alien_Children_Program.csv"
raw = pd.read_csv(raw_path, dtype=str, encoding="utf-8-sig")
raw_audit = validate_capacity_data(raw)
print(raw_audit.report.to_dict())
raw_audit.report.to_frame()

In [ ]:
prepared = preprocess_data(raw_path)
cleaned = prepared.data
print({
    "source_rows": prepared.report.source_rows,
    "cleaned_daily_rows": len(cleaned),
    "period_start": cleaned.index.min().date().isoformat(),
    "period_end": cleaned.index.max().date().isoformat(),
    "dates_inserted": prepared.report.missing_dates_inserted,
    "values_imputed": prepared.report.numeric_values_imputed,
    "logical_anomaly_rows": prepared.report.logical_anomaly_rows,
})
prepared.report.to_frame()

In [ ]:
assert cleaned.index.name == DATE_COLUMN
assert cleaned.index.is_monotonic_increasing
assert not cleaned.index.has_duplicates
assert cleaned.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all()
print("Daily continuity checks passed.")

## Interpretation

Review error-severity findings before analysis. Inserted dates and imputed
values are explicitly flagged so downstream results can be filtered or stress
tested rather than silently accepted.